# Solution to Exercise set 7

There are two main goals for this exercise:

1. To develop optimised classifiers (e.g., a decision tree) using cross-validation, and gain experience with the assessment of classifiers.
2. To gain practical experience with signal processing techniques used for preprocessing, for instance, of Near-Infrared (NIR) spectra. Preprocessing methods are important for improving the signal-to-noise ratio, correcting for scattering effects (variations in light path due to particle size, etc.), and enhancing spectral features, which can lead to more reliable analysis and development of robust predictive models.

**Learning Objectives:**

After completing this exercise set, you will be able to:

- Develop optimised classifiers and assess them.
- Preprocess spectra by normalisation, multiplicative scatter correction, or taking a derivative.

**To get the exercise approved, complete the following problems:**

- [7.1(a)](#7.1(a)), [7.1(b)](#7.1(b)) and [7.1(c)](#7.1(c)): To show that you can create a optimised classifier.
- [7.2(a)](#7.2(a)) and at least one of [7.2(b)](#7.2(b)), [7.2(c)](#7.2(c)) or [7.2(d)](#7.2(d)): To show that you can apply preprocessing to NIR spectra.

**Files required for this exercise:**
* [Exercise 7.1](#Exercise-7.1-Developing-optimised-classifiers): [bace-small.csv](bace-small.csv)
* [Exercise 7.2](#Exercise-7.2-Preprocessing-NIR-spectra): [nir.csv](nir.csv)

Please ensure that these files are saved in the same directory as this notebook.


**Note:** Several of the methods used in this solution are stochastic (e.g., Random Forest and AutoML). You might see different answers (both in your own work and in different runs). This can also lead to minor discrepancies between the text answers and the output from the code cells.

## Exercise 7.1 Developing optimised classifiers

We will here consider a version of the **BACE** dataset from [MoleculeNet](https://moleculenet.org) (site containing benchmark data for molecular machine learning).

This data set contains 1513 molecules that have been labelled by their binding affinity to BACE-1 (1 for active, 0 for inactive). Active binders of BACE-1 could potentially be used as treatments for Alzheimer’s Disease.

The version we consider contains a subset (9) of all features in the original data (around 590):
* `MW`: The total mass (molecular weight) of the molecule.
* `AlogP`: The partition coefficient. Measures the lipophilicity (how much the molecule prefers oil over water).
* `HBA`: Hydrogen Bond Acceptors, the number of atoms that can receive a hydrogen bond.
* `HBD`: Hydrogen Bond Donors, the number of atoms that can be "donated" to form a hydrogen bond.
* `PSA`: Polar Surface Area, the total surface area contributed by polar atoms.
* `RB`: Rotatable Bonds, A count of single bonds that can rotate freely.
* `HeavyAtomCount`: The total number of atoms in the molecule, excluding hydrogen.
* `ChiralCenterCount`: The number of chiral centres in the molecule.
* `RingCount`: The number of cyclic structures (like benzene rings) in the molecule.

The data can be loaded as follows:

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

%matplotlib inline
sns.set_theme(style="ticks", context="notebook", palette="colorblind")

data = pd.read_csv("bace-small.csv")
skip = ("mol", "Class")
y = data["Class"]  # Classification of samples.
class_names = ["Inactive", "Active"]
features = [i for i in data.columns if i not in skip]
print("Features:", features)
X = data[features].to_numpy()
data.head()

### 7.1(a)

**Task: In the following task, you will develop a classifier to predict whether a small molecule is an active (positive) or inactive (negative) binder. Which error type (false positive or false negative) should be minimised?**

#### Your answer to question 7.1(a): Will you minimise false positives or negatives?

The choice between minimising false positives and false negatives depends on the context. Here, the context is not really specified, so we will make our own assumptions about this. We will assume that our (relatively simple) model will be used at an early stage in the drug development. We consider the two types of mistakes:

* False positives ("false alarms"): This corresponds to an inactive molecule being classified as an active molecule. This represents a waste of resources (time and money) since these molecules will be tested further in experiments to validate that they are active.

* False negatives ("missed opportunities"): This corresponds to a truly active molecule being classified as inactive. We would not like to miss any potential drug molecule, since this means discarding that candidate without evaluating it further.

Using the assumption that our modelling work is used in an early stage, perhaps for virtual screening, we prioritise minimising the **false negatives**. This assumes that we will do further work using our classification results, for instance, more advanced modelling or simple experiments to filter out the "false alarms". Therefore, we will maximise the **recall** (this minimises the false negatives) to make sure we miss as few opportunities as possible.

### 7.1(b)

**Task: Prepare the data by splitting it into train/test sets and standardise the features. Use the code provided below and note the order of operations.**


**Hint:**

1. Use the code provided in the cell below to create the training and test sets. We use something called **stratification** here. This makes sure that the train/test split maintains the same proportion of classes as the original dataset.

In [ ]:
# To create the training set use:
from sklearn.model_selection import train_test_split

X_train0, X_test0, y_train, y_test = train_test_split(
    X, y, stratify=y, test_size=0.33, random_state=2026
)

In [ ]:
# Just to get the sizes:
print("Training size:", X_train0.shape)
print("Testing size:", X_test0.shape)
print("\nItems in the training set:", y_train.value_counts())
print("\nItems in the testing set:", y_test.value_counts())

2. Preprocess the data using the [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) from scikit-learn:

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train0)

X_train = scaler.transform(X_train0)
X_test = scaler.transform(X_test0)

#### Your answer to question 7.1(b): Can you give a reason why we fit the `StandardScaler` to the training data and not to the full data set?

We keep the training set separated from the test set to make sure that we do not have [data leakage](https://en.wikipedia.org/wiki/Leakage_(machine_learning)). Therefore, we should always split the data before any preprocessing. The `StandardScaler` is fitted to the training data so that it learns the mean and standard deviation of our training data alone. Once it is fitted, we can use it to scale both the training and testing data.

A practical way to think about this: Say we train our model on data we have today and then, a week later, we collect new data for testing. We can not go back in time to scale the full data we have now, and we have to use the scaling we defined when we trained our model a week ago. By fitting the scaler to the training data only, we simulate this real-world behaviour.

### 7.1(c)

**Task: Create a decision tree classifier to classify molecules. Optimise the tree depth using cross-validation on a training set. Report the optimal maximum depth of the resulting tree.**

With reference to the previous problem:

* If you prioritised minimising false positives, use the `precision` as your optimisation metric.
* If you prioritised minimising false negatives, use the `recall` as your optimisation metric.
* If you opted for a balanced approach, use the `balanced_accuracy` as your optimisation metric.


**Hint:**
1. The optimisation of the decision tree can be done as follows (assuming that you have already split into the training and test sets):


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Set up a grid search:
parameters = {"max_depth": range(1, 10)}
grid_t = GridSearchCV(
    DecisionTreeClassifier(),
    parameters,
    scoring="recall",
    refit=True,
)
# Run the grid search:
grid_t.fit(X_train, y_train)

# Get the best classifier from the grid search:
best_tree = grid_t.best_estimator_
print("Best tree:", best_tree)
print("Best score", grid_t.best_score_)
print("Best parameters", grid_t.best_params_)
print("Actual depth of the tree:", best_tree.get_depth())
# Note: The max_depth variables is just the maximum depth,
# the resulting tree can be shorter if adding more levels
# does not improve the classification.

#### Your answer to question 7.1(c): What depth did you get for your tree?

We got a depth of 1 for the decision tree, while the recall is OK (but not great). We will have to investigate how the tree is making its decision and how well it performs for the test set to further assess it.

**Note:** In the setup above, we have some form of data leakage when we are doing cross-validation (since we use the already scaled training set). It would be better to include the scaling as part of the cross-validation. It is somewhat more complicated, but this can be done as follows:

In [ ]:
from sklearn.pipeline import Pipeline

# Define a pipeline for the model in two steps:
pipe = Pipeline(
    [("scaler", StandardScaler()), ("tree", DecisionTreeClassifier())]
)
# Set up a grid search:
parameters = {"tree__max_depth": range(1, 10)}
grid_tp = GridSearchCV(
    pipe,
    parameters,
    scoring="recall",
    refit=True,
)
# Run the grid search:
grid_tp.fit(X_train0, y_train)

# Get the best classifier from the grid search:
best_tree_2 = grid_tp.best_estimator_.named_steps["tree"]
print("Best tree:", best_tree_2)
print("Best score", grid_tp.best_score_)
print("Best parameters", grid_tp.best_params_)
print("Actual depth of the tree:", best_tree_2.get_depth())
# Note: The max_depth variables is just the maximum depth,
# the resulting tree can be shorter if adding more levels
# does not improve the classification.

We have opted for a simpler approach here, since we will use identical training and test sets for several models.

### 7.1(d)

**Task: Visualise your decision tree and use this to describe how the classification is made.**

**Hint:** The decision tree can be visualised using [plot_tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html) or [export_graphviz](https://scikit-learn.org/stable/modules/generated/sklearn.tree.export_graphviz.html),

1. Easiest: Using [plot_tree](https://scikit-learn.org/stable/modules/generated/sklearn.tree.plot_tree.html):

```python
from sklearn import tree


tree.plot_tree(
    best_tree,  # The tree to plot
    filled=True,  # Add colour to the boxes.
    feature_names=features,  # Get name for features.
    class_names=class_names,  # Get the name of the different classes.
)
```

2. Looks nicer: Using [export_graphviz](https://scikit-learn.org/stable/modules/generated/sklearn.tree.export_graphviz.html):

```python
from sklearn.tree import export_graphviz  # To create the tree.
import graphviz  # To turn the three into a graph, you may need to install this (pip install graphviz).
from IPython.display import display  # To show the graph.

dot_data = export_graphviz(
    best_tree,  # The tree to plot.
    out_file=None,  # Do not write to file.
    feature_names=features,  # Get name for features.
    class_names=class_names,  # Get the name of the different classes.
    rounded=True,  # Show the boxes in the tree with rounded corners.
    filled=True,  # Add colour to the boxes.
)
display(graphviz.Source(dot_data))  # Show the tree in a notebook.
```

In [ ]:
from sklearn.tree import export_graphviz  # To create the tree.
import graphviz  # To turn the three into a graph, you may need to install this (pip install graphviz).
from IPython.display import display  # To show the graph.

dot_data = export_graphviz(
    best_tree,  # The tree to plot.
    out_file=None,  # Do not write to file.
    feature_names=features,  # Get name for features.
    class_names=class_names,  # Get the name of the different classes.
    rounded=True,  # Show the boxes in the tree with rounded corners.
    filled=True,  # Add colour to the boxes.
)
display(graphviz.Source(dot_data))  # Show the tree in a notebook.

#### Your answer to question 7.1(d): What features is the best decision tree using?

Our tree is quite short (to continue with the tree analogy, it seems like a stump). The active branch has a very high (Gini) impurity (0.499), and it is close to the worst-case value of 0.5. This means that the model behaves almost randomly for molecules it labels as active (you could do equally well by flipping a coin). While this split captures the majority of the active molecules (achieving high recall), it fails to separate them from the inactives, resulting in a high number of false positives.

### 7.1(e)

**Task: Create a k-nearest neighbours classifier to classify the molecules. Optimise the number of neighbours using cross-validation on a training set. Report the optimal number of neighbours.**

**Hint:**

1. The optimisation of the k-nearest neighbours classifier can be done as follows (assuming that you have already split into the training and test sets):

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier

# Set up a grid search:
parameters = {"n_neighbors": range(1, 20)}
grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    parameters,
    scoring="recall",  # Swap this with the metric you prefer
)
# Run the grid search:
grid_knn.fit(X_train, y_train)

# Get the best classifier from the grid search:
best_knn = grid_knn.best_estimator_
print("Best knn:", best_knn)
print("Best score", grid_knn.best_score_)
print("Best parameters", grid_knn.best_params_)

#### Your answer to question 7.1(e): What was the optimal number of neighbours?

The optimal number of neighbours was 1. This is typically seen as a **red flag**  since it indicates that the model is probably overfitting. We shall investigate this when we look into the metrics in [7.1(g)](#7.1(g)).

### 7.1(f)

**Task: Create a random forest classifier to classify molecules. Optimise the number of trees and levels using cross-validation on a training set. Report the optimal number of trees and levels.**

**Hint:**

1. The optimisation of the random forest classifier can be done similarly to what you did in [7.1(c)](#7.1(c)) and [7.1(e)](#7.1(e)). You just have to make use of the [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html) and optimise the parameters `n_estimators` and `max_depth`:
```python
from sklearn.ensemble import RandomForestClassifier

# Set up a grid search:
parameters = {
    "n_estimators": [10, 50, 100, 200, 500],  # the number of trees
    "max_depth": range(1, 11),  # the maximum depth
}
grid = GridSearchCV(
    RandomForestClassifier(),
    parameters,
    scoring="recall",  # Swap this with the metric you prefer
    verbose=2,  # Print out text to show the progress of the fitting
)

# ... rest of the optimisation code ...
```

In [ ]:
from sklearn.ensemble import RandomForestClassifier

parameters = {
    "n_estimators": [10, 50, 100, 200, 500],  # the number of trees
    "max_depth": range(1, 11),  # the maximum depth
}
grid_f = GridSearchCV(
    RandomForestClassifier(random_state=2026),
    parameters,
    scoring="recall",  # Swap this with the metric you prefer
    verbose=3,  # Print out text to show the progress of the fitting
    n_jobs=-1,  # Use all processors
)
# Run the grid search:
grid_f.fit(X_train, y_train)

# Get the best classifier from the grid search:
best_forest = grid_f.best_estimator_
print("Best Random forest classifier:", best_forest)
print("Best score", grid_f.best_score_)
print("Best parameters", grid_f.best_params_)

#### Your answer to question 7.1(f): What was the optimal number of estimators and tree depth?

The best tree used 500 estimators and a maximum depth of 10.

**Note:** These numbers will vary since the random forest, as the name suggests, includes some randomness.

### 7.1(g)

**Task: Compare the three optimised classifiers you have made by applying them to the test set and obtaining the corresponding confusion matrices. Also compute the [precision](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_score.html), [recall](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.recall_score.html), and the [balanced accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.balanced_accuracy_score.html) for the test set. Which classifier performs best?**



**Hint:** The metrics can be computed as follows:
```python
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import (
    recall_score,
    precision_score,
    balanced_accuracy_score,
)

y_hat = best_tree.predict(X_test)
recall_tree = recall_score(y_test, y_hat)
precision_tree = precision_score(y_test, y_hat)
bac_tree = balanced_accuracy_score(y_test, y_hat)
print(f"Recall: {recall_tree:.3f}")
print(f"Precision: {precision_tree:.3f}")
print(f"Balanced accuracy: {bac_tree:.3f}")

ConfusionMatrixDisplay.from_estimator(
    best_tree,
    X_test,
    y_test,
    colorbar=True,
)
```

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.metrics import (
    recall_score,
    precision_score,
    balanced_accuracy_score,
)


def score_model(model, X, y_true, ax=None, table=None, add_colorbar=False):
    """Helper method to score classification models."""
    y_pred = model.predict(X)
    recall = recall_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred)
    bac = balanced_accuracy_score(y_true, y_pred)
    name = str(model)
    results = {
        "Classifier": name,
        "Recall": recall,
        "Precision": precision,
        "Balanced accuracy": bac,
    }

    print(f"Recall: {recall:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Balanced accuracy: {bac:.3f}")
    if ax is not None:
        # Plot confusion matrix
        ConfusionMatrixDisplay.from_estimator(
            model,
            X,
            y_true,
            colorbar=False,
            ax=ax,
            im_kw={"vmin": 0, "vmax": y_true.value_counts().max()},
        )
        # NOTE: im_kw sets vmax to a fixed value, this
        # should be set to ensure that it is not smaller than the
        # number of samples in any class.
        ax.set_title(name[:12] + "...", loc="left")
        if add_colorbar:
            fig = ax.get_figure()
            fig.colorbar(ax.images[0], ax=ax, shrink=0.9)
    if table is not None:
        # Store the results in the given table dictionary
        for key in results:
            if key not in table:
                table[key] = []
            table[key].append(results[key])
    return results

In [ ]:
table = {}
fig, axes = plt.subplots(constrained_layout=True, ncols=3, figsize=(9, 3))


for i, model in enumerate((best_tree, best_knn, best_forest)):
    name = str(model)
    print(f"\nScores (Test) for: {model}")
    scores = score_model(
        model, X_test, y_test, ax=axes[i], table=table, add_colorbar=i == 2
    )

table = pd.DataFrame(table)
table.sort_values(by="Recall", ascending=False)

#### Your answer to question 7.1(g): Which classifier performs best?

The decision tree classifier has the highest recall, but also the lowest precision. It captures almost all active molecules, but it produces many false positives. The random forest and the kNN classifiers have more similar scores for the precision and recall (in the range 0.70 to 0.75). These models are still producing errors, but the random forest has a higher recall than the kNN classifier (0.75 vs. 0.72). Therefore, the random forest is the better choice as it has higher recall and balanced accuracy.

Another point in favour of the random forest classifier compared to kNN is that kNN uses only one neighbour. This could lead to problems if we encounter "new" molecules in the future that are significantly different from the rest of the molecules. Since the random forest aggregates classification from multiple trees, we expect this to be more robust than kNN.

### 7.1(h)

Explore if you can create an even better classifier using a gradient boosting method (e.g., [XGBoost](https://xgboost.readthedocs.io)) or a foundation model (e.g., [TabPFN](https://github.com/PriorLabs/TabPFN); note that this requires some extra steps for the installation).

In [ ]:
# Use standard XGBoost:
from xgboost import XGBClassifier

xgb_model = XGBClassifier()
xgb_model.fit(X_train, y_train)

In [ ]:
# Try to automatically find a better classifier:
from flaml import AutoML
import warnings

# This ignores some warning messages we can get while running AutoML
warnings.filterwarnings("ignore")

automl = AutoML()

settings = {
    "time_budget": 300,  # Use 5 minutes
    "metric": "log_loss",
    "verbose": 3,  # Set this to 1 or 2 if it prints too much
    "log_file_name": "fitting-bace.log",
}

automl.fit(X_train, y_train, task="classification", **settings)

In [ ]:
# Store the best model:
model_automl = automl.model.estimator

In [ ]:
from tabpfn import TabPFNClassifier

# Preferably, this should be executed on a GPU, but if we set
# ignore_pretraining_limits=True, we can run it on a CPU:

tabpfn_model = TabPFNClassifier(ignore_pretraining_limits=True).fit(
    X_train, y_train
)

In [ ]:
# To compare everything, we make a figure:
fig, axes = plt.subplots(constrained_layout=True, ncols=6, figsize=(18, 3))

table = {}

for i, model in enumerate(
    (best_tree, best_knn, best_forest, xgb_model, model_automl, tabpfn_model)
):
    name = str(model)
    print(f"\nScores (Test) for: {model}")
    scores = score_model(
        model, X_test, y_test, ax=axes[i], table=table, add_colorbar=i == 5
    )

In [ ]:
table_print = pd.DataFrame(table)
table_print.sort_values(by="Recall", ascending=False)

#### Your answer to question 7.1(h): Do you get better performance?

Yes, we can get a better performance, for instance, with TabPFN (recall of 0.82 and a
precision of 0.78). A comparison of the confusion matrices shows that there are small differences between the best classifiers. This suggests that the features we are currently using may not fully capture what makes molecules active or inactive.

In case you have not executed all the code above, here are the confusion matrices:

![Confusion matrices](results7.1.png)

and a table of the numerical results:

| Classifier                                                |   Recall |   Precision |   Balanced accuracy |
|:----------------------------------------------------------|---------:|------------:|--------------------:|
| DecisionTreeClassifier(max_depth=1)                       | 0.938596 |    0.519417 |            0.605328 |
| KNeighborsClassifier(n_neighbors=1)                       | 0.719298 |    0.709957 |            0.736487 |
| RandomForestClassifier(max_depth=10, n_estimators=500)    | 0.754386 |    0.725738 |            0.757708 |
| XGBClassifier(...)                                        | 0.732456 |    0.719828 |            0.746743 |
| ExtraTreesClassifier(...)                                 | 0.767544 |    0.76087  |            0.782669 |
| TabPFNClassifier()                                        | 0.820175 |    0.779167 |            0.812661 |


The full [BACE data set](https://moleculenet.org/datasets-1) contains over 500 additional features. The next step could be to make use of all features and see if this improves our modelling work!

## Exercise 7.2 Preprocessing NIR spectra

We will analyse NIR spectra from two distinct Ethiopian [sorghum](https://en.wikipedia.org/wiki/Sorghum) cultivars to determine if they can be differentiated. Specifically, we will examine how different preprocessing techniques impact the outcome of a principal component analysis (PCA) applied to the spectra. 

**Note:**

1. The dataset used in this exercise is derived from [Kosmowski and Worku
](https://doi.org/10.1371/journal.pone.0193620) who used a miniaturised NIR spectrometer to identify Ethiopian crop cultivars. To simplify the analysis, we focus on measurements from only two of the ten sorghum cultivars studied in the original work

2. This exercise will mainly ask you to run and observe results from already implemented code.

### 7.2(a)

The following code performs these steps:

1. Load the NIR spectra from the data file [nir.csv](./nir.csv).
2. Extracts wavelengths, spectra, and cultivar names.
3. Defines colours for plotting cultivars.
4. Creates a function to plot spectra by cultivar.
5. Creates a function to run a PCA on provided spectra and plot the scores of the first two principal components.
6. Initialises a figure for results.
7. Plots the original spectra and the PCA results.

**Task: Execute the code and observe the generated plot. In the PCA scores plot, are there any noticeable groupings that suggest cultivar separation?**

In [ ]:
# Load the needed libraries
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.decomposition import PCA

%matplotlib inline
sns.set_theme(style="ticks", context="notebook", palette="colorblind")

# Load the raw data:
data = pd.read_csv("nir.csv")
data.head()

In [ ]:
# Extract information from the data
variables = [i for i in data.columns if i != "Cultivator"]
# Wavelengths as numbers:
wavelengths = np.array([float(i) for i in variables])
print(f"Number of wavelengths: {len(wavelengths)}")
# All spectra as a data matrix:
all_spectra = data[variables].to_numpy()
print(f"Size of data matrix: {all_spectra.shape}")
# Name of the two cultivators:
cultivators = data["Cultivator"].unique()
print(f"Cultivators: {cultivators}")

In [ ]:
# Define a colour mapping for the two cultivators:
colors = sns.color_palette("colorblind", n_colors=len(cultivators))
color_mapping = {key: colori for key, colori in zip(cultivators, colors)}
# Show the two colors
colors

In [ ]:
def plot_spectra(data, X, wavelengths, color_mapping, axi, legend=False):
    """

    Plots NIR spectra from the given data matrix X, colour-coded by cultivar.

    Args:
        data (pandas.DataFrame): DataFrame containing cultivar information.
        X (numpy.ndarray): Matrix of NIR spectra, where each row is a spectrum.
        wavelengths (numpy.ndarray): Array of corresponding wavelengths for the spectra.
        color_mapping (dict): Dictionary mapping cultivar names to colours.
        axi (matplotlib.axes.Axes): Matplotlib Axes object for plotting.
        legend (bool, optional): Whether to include a legend. Defaults to False.

    Returns:
        None (plots directly to the provided Axes object).
    """
    # Initialise empty lists to store legend handles and labels:
    handles, labels = [], []
    for cultivator in color_mapping.keys():
        # Filter spectra belonging to the current cultivar
        spectra_cult = X[data["Cultivator"] == cultivator]
        color = color_mapping[cultivator]
        for spectrum in spectra_cult:
            # Plot each spectrum with the assigned color
            (linei,) = axi.plot(wavelengths, spectrum, color=color)
        # Append the line handle and cultivar label for the legend
        handles.append(linei)
        labels.append(cultivator)
    if legend:
        # Add a legend to the plot if 'legend' is True
        legend = axi.legend(handles, labels, title="Cultivator:")

In [ ]:
def run_pca_plot_scores(data, X, color_mapping, axi):
    """
    Performs Principal Component Analysis (PCA) on the input spectra and plots the scores (colour-coded).

    Args:
        data (pandas.DataFrame): DataFrame containing cultivar information.
        X (numpy.ndarray): Matrix of NIR spectra, where each row is a spectrum.
        color_mapping (dict): Dictionary mapping cultivar names to colours.
        axi (matplotlib.axes.Axes): Matplotlib Axes object for plotting.

    Returns:
        None (plots directly to the provided Axes object).
    """
    pca = PCA(n_components=2)  # Initialize PCA with 2 components
    scores = pca.fit_transform(X)  # Perform PCA and get the scores
    sns.scatterplot(
        data=data,
        x=scores[:, 0],
        y=scores[:, 1],
        hue="Cultivator",
        palette=color_mapping,
        legend=False,
        ax=axi,
    )
    # Calculate explained variance ratios
    perc = pca.explained_variance_ratio_ * 100
    # Set axis labels with explained variance percentages
    axi.set_xlabel(f"Scores PC1 ({perc[0]:.2f}%)")
    axi.set_ylabel(f"Scores PC2 ({perc[1]:.2f}%)")

In [ ]:
figure1, axes1 = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))

plot_spectra(
    data,
    all_spectra,
    wavelengths,
    color_mapping,
    axes1[0],
    legend=True,
)
run_pca_plot_scores(data, all_spectra, color_mapping, axes1[1])

axes1[0].set_xlabel("Wavelength (nm)")
axes1[0].set_ylabel("Absorbance")
axes1[0].set_title("Original spectra", loc="left")
axes1[1].set_title("PCA, Original spectra", loc="left")
sns.despine(fig=figure1)

#### Your answer to question 7.2(a): Is there a clear cultivar separation in the scores plot?

There is some separation in the scores plot, but the cultivars are not fully separated (they overlap).

### 7.2(b)

**Task: Observe the impact of normalisation on the spectra and PCA results. In the PCA scores plot, are there any noticeable groupings that suggest cultivar separation?**

**Hint:**
1. Apply one of the provided normalisations to scale the spectra, for instance
```python
normed = normalise_spectra(all_spectra)
```
2. Plot the normalised spectra and the corresponding PCA results side-by-side. For instance,
```python
figure2, axes2 = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))
plot_spectra(data, normed, wavelengths, color_mapping, axes2[0], legend=True)
run_pca_plot_scores(data, normed, color_mapping, axes2[1])
```

In [ ]:
from sklearn.preprocessing import Normalizer


def normalise_spectra(spectra):
    """Normalise the given spectra to the range [-1, 1]."""
    s_min = spectra.min(axis=1, keepdims=True)
    s_max = spectra.max(axis=1, keepdims=True)
    return 2 * (spectra - s_min) / (s_max - s_min) - 1


def vector_norm(spectra):
    """Norm each row to a length of 1."""
    scaler = Normalizer(norm="l2")
    return scaler.fit_transform(spectra)


def snv(spectra):
    """Normalise by standardising each row: (x - mean) / std, per row."""
    return (spectra - np.mean(spectra, axis=1, keepdims=True)) / np.std(
        spectra, axis=1, keepdims=True
    )


def max_peak_normalisation(spectra):
    """Normalise spectra so that the max for each spectrum is at 1."""
    return spectra / np.max(spectra, axis=1, keepdims=True)

In [ ]:
normed_min_max = normalise_spectra(all_spectra)
normed_vector = vector_norm(all_spectra)
normed_snv = snv(all_spectra)
normed_peak = max_peak_normalisation(all_spectra)

figure2, axes2 = plt.subplots(
    constrained_layout=True, ncols=4, nrows=2, figsize=(12, 5)
)

norms = [normed_min_max, normed_vector, normed_snv, normed_peak]
text = ["Normed to [-1, 1]", "Normed to length 1", "SNV", "Peak normalisation"]

for i, normi in enumerate(norms):
    axes2[0, i].set_title(text[i], loc="left")
    plot_spectra(
        data, normi, wavelengths, color_mapping, axes2[0, i], legend=i == 0
    )
    run_pca_plot_scores(data, normi, color_mapping, axes2[1, i])
sns.despine(fig=figure2)

#### Your answer to question 7.2(b): Is there a clear cultivar separation in the scores plot?

Yes, the separation is much clearer now. For instance, when norming each spectrum to the range -1 to 1, the `Teshale` samples form a group on the left side and `Gambella_1107` can be found on the right-hand side. There is some overlap between the groups, but the separation is significantly improved compared to the unprocessed data in ([7.2(a)](#7.2(a))).

### 7.2(c)

**Task: Observe the impact of multiplicative scatter correction (MSC) on the spectra and PCA results. In the PCA scores plot, are there any noticeable groupings that suggest cultivar separation?**

**Hint:**
1. Apply the provided MSC function to correct the spectra, for instance,
```python
corrected = multiplicative_scatter_correction(all_spectra)
```
2. Plot the corrected spectra and the corresponding PCA results side-by-side. For instance,
```python
figure3, axes3 = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))
plot_spectra(data, corrected, wavelengths, color_mapping, axes3[0], legend=True)
run_pca_plot_scores(data, corrected, color_mapping, axes3[1])
```

In [ ]:
def multiplicative_scatter_correction(spectra):
    """
    Applies Multiplicative Scatter Correction (MSC) to the input spectra.

    MSC is a preprocessing technique used to reduce the effects of scatter in spectral data.
    It corrects for variations in path length and particle size, which can affect the
    baseline and slope of the spectra.

    Args:
        spectra (numpy.ndarray): Matrix of spectra, where each row is a spectrum.

    Returns:
        numpy.ndarray: MSC-corrected spectra matrix.
    """

    mean = np.mean(spectra, axis=0)  # Calculate the mean spectrum
    msc_spectra = np.zeros_like(
        spectra
    )  # Initialise an array to store MSC-corrected spectra
    for i, spectrum in enumerate(spectra):
        # Fit a linear regression model to each spectrum against the mean spectrum
        param = np.polyfit(mean, spectrum, 1)
        # Apply the MSC correction: (spectrum - intercept) / slope
        msc_spectra[i] = (spectrum - param[1]) / (param[0])
    return msc_spectra

In [ ]:
corrected = multiplicative_scatter_correction(all_spectra)
figure3, axes3 = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))
plot_spectra(
    data, corrected, wavelengths, color_mapping, axes3[0], legend=True
)
run_pca_plot_scores(data, corrected, color_mapping, axes3[1])
sns.despine(fig=figure3)

#### Your answer to question 7.2(c): Is there a clear cultivar separation in the scores plot?

Yes, the results are similar to the results in [7.2(b)](#7.2(b)): We see some separation between `Teshale` and `Gambella_1107` samples. There is still some overlap between the groups, but the separation is significantly improved compared to the unprocessed data in [7.2(a)](#7.2(a)).

### 7.2(d)

**Task: Investigate the impact of applying a second derivative transformation on the spectra and PCA results. In the PCA scores plot, are there any noticeable groupings that suggest cultivar separation?**

**Hint:**
1. Use the provided code to calculate the second derivative of the original spectra, for instance,

```python
dspectra = derivative(wavelengths, all_spectra, deriv=2)
```
2. Plot the resulting second derivative spectra and the corresponding PCA results side-by-side. For instance,
```python
figure4, axes4 = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))
plot_spectra(data, dspectra, wavelengths, color_mapping, axes4[0], legend=True)
run_pca_plot_scores(data, dspectra, color_mapping, axes4[1])
```

**Note:** The derivative is computed using the [Savitzky-Golay filter](https://en.wikipedia.org/wiki/Savitzky%E2%80%93Golay_filter). This method smooths the data by fitting a polynomial to a moving window of points and then calculating the derivative of that fitted polynomial. The method, as implemented here, assumes evenly spaced data points. It may produce inaccurate results if your wavelengths are unevenly spaced. In such cases, alternative methods like B-spline derivatives or other interpolation-based approaches might be more suitable.

In [ ]:
from scipy.signal import savgol_filter


def derivative(wavelengths, spectra, window_length=21, polyorder=3, deriv=2):
    """
    Calculates the derivative of the input spectra using the Savitzky-Golay filter.

    This function applies the Savitzky-Golay filter to smooth and differentiate the
    input spectra. The filter is used to reduce noise and enhance spectral features.

    Args:
        wavelengths (numpy.ndarray): Array of wavelengths corresponding to the spectra.
        spectra (numpy.ndarray): Matrix of spectra, where each row is a spectrum.
        window_length (int): The length of the filter window (must be odd).
        polyorder (int): The order of the polynomial used to fit the samples.
        deriv (int, optional): The order of the derivative to compute. Defaults to 2 (second derivative).

    Returns:
        numpy.ndarray: Matrix of derivative spectra.
    """
    # Apply Savitzky-Golay filter to calculate the derivative:
    delta_w = wavelengths[1] - wavelengths[0]
    dspectra = savgol_filter(
        spectra,
        window_length,
        polyorder,
        deriv=deriv,
        delta=delta_w,  # Wavelength spacing
        mode="nearest",  # Extrapolation mode at the edges
        axis=1,  # Process each row
    )
    return dspectra

In [ ]:
dspectra = derivative(wavelengths, all_spectra, deriv=2)
figure4, axes4 = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))
plot_spectra(data, dspectra, wavelengths, color_mapping, axes4[0], legend=True)
run_pca_plot_scores(data, dspectra, color_mapping, axes4[1])
sns.despine(fig=figure4)

#### Your answer to question 7.2(d): Is there a clear cultivar separation in the scores plot?

Yes, the different cultivars separate into three well-defined clusters. The separation is significantly clearer than in the preceding cases.

### 7.2(e)

**Task: Explain how the Savitzky-Golay filter uses polynomial fitting to smooth data and compute derivatives.**

**Hint:** See page 149 in our textbook.

#### Your answer to question 7.2(e): Your explanation for Savitzky-Golay filtering?

The Savitzky-Golay filter smooths data by operating on a moving window of data points. Within each window, it fits a polynomial curve to the data using least squares. The centre point of the window is then replaced with the corresponding value from the fitted polynomial, effectively smoothing the signal. Derivatives are calculated directly from the fitted polynomial within each window.

### 7.2(f)

**Task: The figure below displays the results of the preprocessing steps from exercise [7.2(a)](#7.2(a)) to [7.2(d)](#7.2(d)). Based on these results, which preprocessing method appears most promising for building a classifier?**

![Preprocessing NIR results](results7.2.png)

#### Your answer to question 7.2(f): Which preprocessing step appears most promising?

The scores plot showing the results when applying the second derivative preprocessing yields three well-separated clusters, corresponding to the different cultivars. This strongly suggests that a pipeline of taking the second derivative followed by PCA would generate scores well-suited for a classification model input. However, without building a classifier, we cannot definitively rule out the potential effectiveness of the other preprocessing approaches (or their combination). Nevertheless, the results strongly indicate the second derivative as the most promising.

### 7.2(g)

**Task: Assuming that the largest variation in the raw data is due to scattering effects, we could assume that PCA should pick up on this in the first (and perhaps second) principal component. If you go back to [7.2(a)](#7.2(a)), will the scores plot look more promising if you use principal components 2 and 3?**

In [ ]:
# We first make the scores plot:
fig, ax = plt.subplots(constrained_layout=True)
pca = PCA(n_components=3)  # Initialize PCA with 3 components
scores = pca.fit_transform(all_spectra)  # Perform PCA and get the scores
sns.scatterplot(
    data=data,
    x=scores[:, 1],
    y=scores[:, 2],
    hue="Cultivator",
    palette=color_mapping,
    legend=False,
    ax=ax,
)
# Calculate explained variance ratios
perc = pca.explained_variance_ratio_ * 100
# Set axis labels with explained variance percentages
ax.set_xlabel(f"Scores PC2 ({perc[1]:.2f}%)")
ax.set_ylabel(f"Scores PC3 ({perc[2]:.2f}%)")
sns.despine(fig=fig)

We see now that the scores plot looks more promising! It looks like PC1 is effectively describing the scattering effects. Let us reproduce the spectra by removing the first principal component:

In [ ]:
scores_zero = scores.copy()
scores_zero[:, 0] = 0

spectra_pca = pca.inverse_transform(scores_zero)

figure5, axes5 = plt.subplots(constrained_layout=True, ncols=2, figsize=(8, 4))


plot_spectra(
    data, spectra_pca, wavelengths, color_mapping, axes5[0], legend=True
)
axes5[0].set_title("PCA reconstructed spectra", loc="left")
run_pca_plot_scores(data, spectra_pca, color_mapping, axes5[1])
sns.despine(fig=figure5)

#### Your answer to question 7.2(g): Will using other principal components improve the separation in the scores plot?

Yes, if we focus on PC2 and PC3 we can improve the separation.

Our assumption when removing scattering effects with MSC was that scattering effects contribute significantly more to the total variance than the underlying chemical signatures. If this is true, and since PCA focus on the largest variance, the first principal component is expected to capture this scattering. Our results support this, and the reconstructed spectra are similar to the normalised (or MSC) spectra.